In [71]:
import os
import cv2
import numpy as np

In [72]:
# Configuration
INPUT_ROOT_DIR = r"C:\Users\Asus TUF -PC\LeafSense AI training\LeafSense"
PROCESSED_DIR = r"C:\Users\Asus TUF -PC\LeafSense AI training\LeafSenseProcessed"
CLASSES = ["ArtocarpusHeterophyllus", "BroussonetiaPapyrifera", "CeibaPentandra", "CocosNucifera", 
           "DurioZibethinus", "ElaeisGuineensis", "EuphorbiaPulcherrima", "ManihotEsculenta", 
           "SamaneaSaman", "TheobromaCacao"]

In [73]:
def create_directories():
    """Create the dataset structure: train, val, test with classes"""
    for split in ["train", "val", "test"]:
        for cls in CLASSES:
            os.makedirs(os.path.join(PROCESSED_DIR, split, cls), exist_ok=True)


In [74]:
def preprocess_image(image_path):
    """Comprehensive image preprocessing using OpenCV"""
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Unable to read image: {image_path}")

    # Resize
    img_resized = cv2.resize(img, (224, 224), interpolation=cv2.INTER_AREA)

    # Noise reduction
    img_denoised = cv2.fastNlMeansDenoisingColored(img_resized, None, 10, 10, 7, 21)

    # Contrast enhancement
    lab = cv2.cvtColor(img_denoised, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    limg = cv2.merge((cl, a, b))
    img_enhanced = cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)

    return img_enhanced

In [75]:
def augment_image(image):
    """Generate augmented versions of an image"""
    augmentations = [image]

    # Horizontal flip
    augmentations.append(cv2.flip(image, 1))

    # Vertical flip
    augmentations.append(cv2.flip(image, 0))

    # Rotations
    rows, cols = image.shape[:2]
    for angle in [15, -15]:
        rotation_matrix = cv2.getRotationMatrix2D((cols / 2, rows / 2), angle, 1)
        augmentations.append(cv2.warpAffine(image, rotation_matrix, (cols, rows)))

    # Brightness variations
    for alpha in [0.7, 1.3]:
        bright = cv2.convertScaleAbs(image, alpha=alpha, beta=0)
        augmentations.append(bright)

    return augmentations

In [76]:
def process_dataset():
    """Process and augment images into the new structure"""
    create_directories()

    aug_counter = {}

    # Iterate over dataset folders (train, val, test)
    for dataset_dir in ["train", "val", "test"]:
        dataset_path = os.path.join(INPUT_ROOT_DIR, "data", dataset_dir)

        if not os.path.exists(dataset_path):
            print(f"Skipping missing dataset folder: {dataset_path}")
            continue

        # Iterate over each class
        for cls in CLASSES:
            cls_path = os.path.join(dataset_path, cls)

            if not os.path.exists(cls_path):
                print(f"Skipping missing folder: {cls_path}")
                continue

            aug_counter[cls] = aug_counter.get(cls, 0)

            # Process each image
            for img_name in os.listdir(cls_path):
                full_path = os.path.join(cls_path, img_name)

                try:
                    processed_img = preprocess_image(full_path)
                    augmented_images = augment_image(processed_img)

                    # Save the augmented images in the correct structure
                    for i, aug_img in enumerate(augmented_images):
                        output_filename = f"{dataset_dir}_{os.path.splitext(img_name)[0]}_aug{i}.jpg"
                        output_dir = os.path.join(PROCESSED_DIR, dataset_dir, cls)
                        output_path = os.path.join(output_dir, output_filename)

                        cv2.imwrite(output_path, aug_img)
                        aug_counter[cls] += 1

                except Exception as e:
                    print(f"Error processing {full_path}: {e}")

    # Print augmentation summary
    print("\nAugmentation Summary:")
    for cls, count in aug_counter.items():
        print(f"{cls}: {count} images")


In [77]:
def verify_dataset():
    """Verify processed dataset"""
    print("\nDataset Verification:")
    for split in ["train", "val", "test"]:
        for cls in CLASSES:
            cls_path = os.path.join(PROCESSED_DIR, split, cls)
            if os.path.exists(cls_path):
                image_count = len([f for f in os.listdir(cls_path) if f.endswith(('.jpg', '.png', '.jpeg'))])
                print(f"{split}/{cls}: {image_count} images")
            else:
                print(f"{split}/{cls}: Folder not found")

# Run the processing
process_dataset()
verify_dataset()

print("\nDataset preprocessing complete. Ready for Roboflow upload!")


Augmentation Summary:
ArtocarpusHeterophyllus: 5649 images
BroussonetiaPapyrifera: 6321 images
CeibaPentandra: 6209 images
CocosNucifera: 6272 images
DurioZibethinus: 6335 images
ElaeisGuineensis: 6244 images
EuphorbiaPulcherrima: 6300 images
ManihotEsculenta: 4928 images
SamaneaSaman: 5047 images
TheobromaCacao: 6265 images

Dataset Verification:
train/ArtocarpusHeterophyllus: 2870 images
train/BroussonetiaPapyrifera: 3500 images
train/CeibaPentandra: 3423 images
train/CocosNucifera: 3472 images
train/DurioZibethinus: 3500 images
train/ElaeisGuineensis: 3479 images
train/EuphorbiaPulcherrima: 3493 images
train/ManihotEsculenta: 3136 images
train/SamaneaSaman: 2233 images
train/TheobromaCacao: 3458 images
val/ArtocarpusHeterophyllus: 2086 images
val/BroussonetiaPapyrifera: 2093 images
val/CeibaPentandra: 2086 images
val/CocosNucifera: 2100 images
val/DurioZibethinus: 2100 images
val/ElaeisGuineensis: 2037 images
val/EuphorbiaPulcherrima: 2100 images
val/ManihotEsculenta: 1071 images
v

Processed 700 augmented images in C:/Users/Asus TUF -PC/LeafSense AI training/LeafSense/data/test/CeibaPentandra
